# 03. Supervised Answer Verification

This notebook trains traditional Machine Learning models to perform Answer Verification.
Given the 20,013-dimensional feature representation of an `(article, question, option)` row, the model predicts if the option is the correct answer (1) or incorrect (0).

## Models Trained
1. Logistic Regression (balanced & unweighted)
2. Linear SVM (balanced & unweighted)
3. Multinomial Naive Bayes
4. Random Forest
5. XGBoost

## Evaluation Metrics
1. **Binary Verification (Row-level):** Accuracy, Macro F1, Precision, Recall.
2. **MCQ Exact Match (Group-level):** For a single question, the model scores all 4 options. The option with the highest score (`predict_proba` or `decision_function`) is selected as the predicted answer. If it matches the true correct option, the MCQ is marked correct.


In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import load_npz
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models_new")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Loading feature matrices and labels...")
X_train = load_npz(PROCESSED_DIR / "X_train_features.npz")
y_train = np.load(PROCESSED_DIR / "y_train.npy")
train_sample_ids = np.load(PROCESSED_DIR / "train_sample_ids.npy", allow_pickle=True)

X_val = load_npz(PROCESSED_DIR / "X_val_features.npz")
y_val = np.load(PROCESSED_DIR / "y_val.npy")
val_sample_ids = np.load(PROCESSED_DIR / "val_sample_ids.npy", allow_pickle=True)

print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Val shapes: X={X_val.shape}, y={y_val.shape}")


Loading feature matrices and labels...
Train shapes: X=(281032, 20013), y=(281032,)
Val shapes: X=(35436, 20013), y=(35436,)


In [2]:
def evaluate_model(model, X, y, sample_ids, model_name):
    """Evaluates a model on row-level metrics and group-level (MCQ) exact match."""
    # Row-level metrics
    y_pred = model.predict(X)
    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred, average='macro')
    prec = precision_score(y, y_pred, average='macro', zero_division=0)
    rec = recall_score(y, y_pred, average='macro', zero_division=0)
    
    # Group-level (MCQ Exact Match) metrics
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X)
    else:
        scores = y_pred
        
    df = pd.DataFrame({'sample_id': sample_ids, 'label': y, 'score': scores})
    
    # For each sample_id, pick the option with max score
    correct_mcqs = 0
    total_mcqs = 0
    
    for group_name, group_df in df.groupby('sample_id'):
        if len(group_df) == 0:
            continue
        total_mcqs += 1
        # Get index of max score
        best_idx = group_df['score'].idxmax()
        # Check if the label at that index is 1
        if group_df.loc[best_idx, 'label'] == 1:
            correct_mcqs += 1
            
    mcq_acc = correct_mcqs / total_mcqs if total_mcqs > 0 else 0
    
    print(f"--- {model_name} ---")
    print(f"Binary Acc: {acc:.4f} | Macro F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")
    print(f"MCQ Exact Match: {mcq_acc:.4f} ({correct_mcqs}/{total_mcqs})\n")
    return mcq_acc


In [3]:
print("Training Logistic Regression (Balanced)...")
lr_balanced = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
lr_balanced.fit(X_train, y_train)
evaluate_model(lr_balanced, X_val, y_val, val_sample_ids, "Logistic Regression (Balanced)")
joblib.dump(lr_balanced, MODELS_DIR / "lr_balanced.pkl")

print("Training Logistic Regression (Unweighted)...")
lr_unweighted = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_unweighted.fit(X_train, y_train)
evaluate_model(lr_unweighted, X_val, y_val, val_sample_ids, "Logistic Regression (Unweighted)")
joblib.dump(lr_unweighted, MODELS_DIR / "lr_unweighted.pkl")


Training Logistic Regression (Balanced)...


c:\AI_Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Logistic Regression (Balanced) ---
Binary Acc: 0.5747 | Macro F1: 0.5251 | Prec: 0.5390 | Rec: 0.5509
MCQ Exact Match: 0.3696 (3274/8859)

Training Logistic Regression (Unweighted)...


c:\AI_Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Logistic Regression (Unweighted) ---
Binary Acc: 0.7505 | Macro F1: 0.4334 | Prec: 0.6825 | Rec: 0.5019
MCQ Exact Match: 0.3692 (3271/8859)



['..\\models_new\\lr_unweighted.pkl']

In [4]:
print("Training Multinomial Naive Bayes...")
nb = MultinomialNB(alpha=1.0)
# Make sure X_train has no negative values. TF-IDF and distances are non-negative.
nb.fit(X_train, y_train)
evaluate_model(nb, X_val, y_val, val_sample_ids, "Multinomial Naive Bayes")
joblib.dump(nb, MODELS_DIR / "naive_bayes.pkl")


Training Multinomial Naive Bayes...
--- Multinomial Naive Bayes ---
Binary Acc: 0.7465 | Macro F1: 0.4514 | Prec: 0.5735 | Rec: 0.5064
MCQ Exact Match: 0.3692 (3271/8859)



['..\\models_new\\naive_bayes.pkl']